In [20]:
import math
import random

In [42]:
class Scalar:
  def __init__(self,data,_children=(),_op=''):
    self.data=data
    self.grad= 0.0
    self._prev=_children
    self._op=_op
    self._backward=lambda: None

  def __repr__(self):
    return f"my scalar value: {self.data} and grad is: {self.grad}"

  def __add__(self,other):
    other= other if isinstance(other, Scalar) else Scalar(other) # Ensure other is Scalar
    out= Scalar(self.data +other.data, _children=(self,other),_op='+')

    def _backward():
      self.grad+=out.grad
      other.grad+=out.grad
    out._backward=_backward
    return out

  def __radd__(self, other): # other + self
      return self + other

  def __iadd__(self, other):
      # Handle in-place addition, directly modifying self.data
      other = other if isinstance(other, Scalar) else Scalar(other)
      self.data += other.data

      return self

  def __sub__(self,other):
    other= other if isinstance(other, Scalar) else Scalar(other) # Ensure other is Scalar
    out= Scalar(self.data - other.data, _children=(self,other),_op='-')
    def _backward():
        self.grad += out.grad
        other.grad -= out.grad # Gradient of subtraction is -1
    out._backward = _backward
    return out

  def __mul__(self,other):
    other= other if isinstance(other, Scalar) else Scalar(other)
    out= Scalar(self.data *other.data, _children=(self,other),_op='*')
    def _backward():
      self.grad+=other.data*out.grad
      other.grad+=self.data*out.grad
    out._backward=_backward
    return out

  def __pow__(self,other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Scalar(self.data**other, (self,), f'**{other}')

    def _backward():
        self.grad += other * (self.data**(other-1)) * out.grad
    out._backward=_backward
    return out

  def __truediv__(self, other):
    # Division = multiplication by reciprocal
    other = other if isinstance(other, Scalar) else Scalar(other) # Ensure other is Scalar
    out = Scalar(self.data / other.data, _children=(self, other), _op='/')
    def _backward():
        self.grad += out.grad / other.data
        other.grad -= out.grad * self.data / (other.data**2)
    out._backward = _backward
    return out

  def ReLu(self):
    out_data=max(0.0,self.data)
    out=Scalar(out_data,(self,),'ReLu')

    def _backward():
      self.grad+=out.grad*(out_data>0)
    out._backward=_backward
    return out

  def tanh(self):
    x= self.data;
    t=(math.exp(2*x)-1)/(math.exp(2*x)+1)
    out=Scalar(t,(self,),'tanh')

    def _backward():
      self.grad+=(1-t**2)*out.grad
    out._backward=_backward
    return out

  def backward(self):
    topo=[]
    visited= set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)

    build_topo(self)

    self.grad=1
    for node in reversed(topo):
      node._backward()
# Scalar._backward = Scalar.backward

In [22]:
import random

In [23]:
class Neuron:
  def __init__(self,num_inputs) -> None:
    self.weights = [Scalar(random.uniform(-1,1)) for _ in range (num_inputs)]
    self.bias = Scalar(random.random())

  def __call__(self,x):
    activation=sum((wi*xi for wi,xi in zip(self.weights,x)),self.bias)
    out=activation.tanh()
    return out

  def parameters(self):
    return self.weights+[self.bias]


In [47]:
class Layer:
  def __init__(self,num_inputs,num_neurons) -> None:
    self.neurons=[Neuron(num_inputs) for _ in range(num_neurons)]

  def __call__(self,x):
    output= [n(x) for n in self.neurons]
    # return output[0] if len(output) == 1 else output # Return the output, handle single neuron case
    return output
  def parameters(self):
    param= [p for neuron in self.neurons for p in neuron.parameters() ]

    # print(f"this layer has {len(param)} params")
    return [p for neuron in self.neurons for p in neuron.parameters() ]

In [25]:
class MLP:
  def __init__(self,num_inputs,num_layers):
    all_sizes= [num_inputs]+ num_layers
    self.layers=[Layer(all_sizes[i],all_sizes[i+1]) for i in range(len(num_layers))]

  def __call__(self,x):
    for layer in self.layers:
      x= layer(x)
    return x

  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]

In [49]:
my_mlp= MLP(3,[4,4,1])
print(f"our mlp has {len(my_mlp.parameters())}")

our mlp has 41


In [50]:
#dataset
xs=[
    [2.0, 3.0, -1],
    [3.0,-1.0,0.5],
    [0.5,1.0,1.0],
    [1.0,1.0, -1.0]
]

ys= [1.0,-1.0,-1.0,1.0]



In [52]:
mlp = MLP(3,[4,4,1])

learning_rate = .1
epochs = 25

for k in range(epochs):
  y_pred = [mlp(x) for x in xs]
  loss_term = [(yout[0] - ygt)**2 for ygt, yout in zip(ys,y_pred)]
  loss = sum(loss_term, Scalar(0.0))

  for p in mlp.parameters():
    p.grad = 0.0
  loss.backward()


  for p in mlp.parameters():
    p.data += -learning_rate * p.grad

  print(f"Epoch : {k+1} and Loss : {loss.data}" )




Epoch : 1 and Loss : 4.989243381196944
Epoch : 2 and Loss : 1.5553751826117659
Epoch : 3 and Loss : 0.331896335862304
Epoch : 4 and Loss : 0.05157535824444015
Epoch : 5 and Loss : 0.041079749317561434
Epoch : 6 and Loss : 0.03445690042103319
Epoch : 7 and Loss : 0.029798582401012144
Epoch : 8 and Loss : 0.02630317343417137
Epoch : 9 and Loss : 0.02356507933408849
Epoch : 10 and Loss : 0.021353179730662666
Epoch : 11 and Loss : 0.019524404983159806
Epoch : 12 and Loss : 0.017984629538162548
Epoch : 13 and Loss : 0.016668975073180988
Epoch : 14 and Loss : 0.015531049199447549
Epoch : 15 and Loss : 0.014536678177816936
Epoch : 16 and Loss : 0.013660064123153438
Epoch : 17 and Loss : 0.012881326991018195
Epoch : 18 and Loss : 0.012184875784246325
Epoch : 19 and Loss : 0.011558296444901743
Epoch : 20 and Loss : 0.010991572786841096
Epoch : 21 and Loss : 0.010476528455886962
Epoch : 22 and Loss : 0.010006419352013116
Epoch : 23 and Loss : 0.009575630785807148
Epoch : 24 and Loss : 0.00917944